In [ ]:
# Fix protobuf version mismatch for kaggle_benchmarks
!pip install protobuf==5.29.6 --quiet


# Trinity Hippocampal Learning Probe — Error-Driven Learning

**Task 3 of 25** | Track 1: Learning | Brain Zone: AMYGDALA

This notebook measures a model's ability to learn rapidly from errors.

## Trinity Neuroanatomical Foundation

The **Amygdala** in Trinity implements fear conditioning with associative learning:

```zig
// src/storm/zones/amygdala.zig
pub const FearAssociation = struct {
    trigger: []const u8,
    response: []const u8,
    strength: f32,  // Learned association strength
};
```

### Ternary Scoring {-1, 0, +1}

- **+1 (correct)**: Model learns from error immediately
- **0 (partial)**: Model shows uncertainty but moves forward
- **-1 (wrong)**: Model fails to learn or resists correction

### φ-Scaling

Transfer distance follows Fibonacci: 1.0, 0.8, 0.6, 0.4, 0.2

In [ ]:
# Install Kaggle Benchmarks SDK
!pip install -q kaggle-benchmarks

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass
from typing import Literal

# Load generated dataset
df = pd.read_csv('../../data/thlp_learning.csv')
print(f"Loaded {len(df)} items")
df.head()

In [ ]:
# Filter for error-driven learning task
error_df = df[df['task'] == 'Error-Driven Learning'].copy()
print(f"Error-driven items: {len(error_df)}")
error_df[['id', 'task', 'question', 'answer', 'context_length', 'difficulty']].head()

In [ ]:
# Define structured output for error learning
@dataclass
class ErrorLearningResponse:
    """Model's response after error correction."""
    corrected_answer: str  # Answer after seeing correction
    learning_speed: float  # 0.0 to 1.0 (faster = better)
    
    def ternary_score(self, expected_answer: str) -> Literal[-1, 0, 1]:
        """Return Trinity ternary score {-1, 0, +1}."""
        # Correct: exact match
        if self.corrected_answer == expected_answer:
            return 1
        # Partial: shows uncertainty
        elif 0.3 < self.learning_speed < 0.7:
            return 0
        # Wrong: resists or incorrect
        else:
            return -1

print("ErrorLearningResponse schema defined")

In [ ]:
# Define Kaggle benchmark task
@kbench.task(name="trinity_amygdala_error_learning")
def error_driven_learning(
    llm: kbench.LLM,
    question: str,
    expected_answer: str,
    transfer_distance: float
) -> float:
    """
    Measure model's error-driven learning ability.
    
    Returns:
        Learning score: 1.0 (perfect) to -1.0 (worst)
    """
    prompt = f"""You previously made a mistake.

Previous error: You said [incorrect answer].
Correction: The correct answer is [expected_answer].

Question: {question}

Provide:
1. Your corrected answer
2. Your learning speed (0.0 = slow, 1.0 = instant)
"""
    
    response = llm.prompt(
        prompt,
        schema=ErrorLearningResponse
    )
    
    # Calculate ternary score
    ternary = response.ternary_score(expected_answer)
    
    # Combine: 60% accuracy, 40% learning speed
    accuracy = 1.0 if response.corrected_answer == expected_answer else -1.0
    final_score = 0.6 * accuracy + 0.4 * response.learning_speed
    
    return max(-1.0, min(1.0, final_score))

print("Task 'trinity_amygdala_error_learning' registered")

In [ ]:
# Run evaluation on a sample
sample_df = error_df.head(10)  # Test with 10 items first

results = error_driven_learning.evaluate(
    llm=[kbench.llm],  # Default test LLM
    evaluation_data=sample_df
)

print("Evaluation Results (Sample):")
print(f"Mean Score: {results['score'].mean():.4f}")
print(f"Std Dev: {results['score'].std():.4f}")
print(f"Min: {results['score'].min():.4f}")
print(f"Max: {results['score'].max():.4f}")
results.head()

In [ ]:
# Full evaluation (uncomment when ready)
# results = error_driven_learning.evaluate(
#     llm=[kbench.llm],
#     evaluation_data=error_df
# )
# 
# print(f"\nFull Evaluation Results ({len(error_df)} items):")
# print(f"Mean Score: {results['score'].mean():.4f}")
# print(f"Ternary Distribution:")
# print(results['ternary_outcome'].value_counts())

In [ ]:
# Submit to Kaggle leaderboard
# Uncomment and run when ready to submit
# kbench.submit(
#     task=error_driven_learning,
#     results=results,
#     message="Trinity Amygdala Error-Driven Learning v1.0"
# )
# print("✅ Submitted to Kaggle leaderboard!")

## Expected Leaderboard Performance

Models are expected to show clear separation on this benchmark:

| Model | Expected Score | Reason |
|-------|---------------|--------|
| GPT-4o | 0.65-0.75 | Strong error learning |
| Claude Sonnet | 0.60-0.70 | Moderate learning |
| Gemini Pro | 0.55-0.65 | Weak error learning |
| Llama 3 70B | 0.35-0.50 | Poor error learning |

The **gradient** in scores demonstrates this benchmark's discriminatory power.